In [ ]:
import pandas as pd
import re
import unicodedata

# ==================================================
# 1. LOAD THE THREE DATASETS
# ==================================================

bushtel = pd.read_csv("bushtel_communities_2024.csv")

abs_population = pd.read_csv(
    "2021Census_I01A_NT_ILOC.csv",
    usecols=[
        "ILOC_CODE_2021",
        "Tot_p_TotP",
        "Tot_p_Indig_P"
    ]
)

abs_geography = pd.read_excel(
    "2021Census_geog_desc_1st_2nd_3rd_release.xlsx",
    sheet_name="2021_ASGS_Indigenous_Structures",
    dtype=str
)


# ==================================================
# 2. RETAIN ONLY ILOC GEOGRAPHIES
# ==================================================

abs_geography = abs_geography[
    abs_geography["ASGS_Structure"] == "ILOC"
].copy()

abs_geography = abs_geography[
    [
        "Census_Code_2021",
        "Census_Name_2021"
    ]
].rename(
    columns={
        "Census_Code_2021": "ILOC_CODE_2021",
        "Census_Name_2021": "iloc_name"
    }
)


# ==================================================
# 3. RENAME THE ABS POPULATION COLUMNS
# ==================================================

abs_population = abs_population.rename(
    columns={
        "Tot_p_TotP": "abs_total_population_2021",
        "Tot_p_Indig_P": "abs_indigenous_population_2021"
    }
)


# ==================================================
# 4. JOIN ABS POPULATION TO ILOC NAMES
# ==================================================

abs_iloc = abs_population.merge(
    abs_geography,
    on="ILOC_CODE_2021",
    how="left",
    validate="one_to_one"
)


# ==================================================
# 5. NORMALISE COMMUNITY NAMES
# ==================================================

def normalize_name(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Convert Unicode characters to standard ASCII
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")

    # Convert to uppercase
    value = value.upper()

    # Replace hyphens with spaces
    value = re.sub(r"[-–—]", " ", value)

    # Remove special characters
    value = re.sub(r"[^A-Z0-9 ]", " ", value)

    # Remove duplicated spaces
    value = re.sub(r"\s+", " ", value).strip()

    return value


bushtel["match_key"] = (
    bushtel["community_name"].apply(normalize_name)
)

abs_iloc["match_key"] = (
    abs_iloc["iloc_name"].apply(normalize_name)
)


# ==================================================
# 6. CHECK FOR DUPLICATED ILOC NAMES
# ==================================================

duplicate_abs_names = abs_iloc[
    abs_iloc["match_key"].duplicated(keep=False)
]

if not duplicate_abs_names.empty:
    print("Warning: duplicated ILOC names were found.")

    print(
        duplicate_abs_names[
            ["ILOC_CODE_2021", "iloc_name"]
        ]
    )


# ==================================================
# 7. JOIN BUSHTEL WITH ABS USING NORMALISED NAMES
# ==================================================

result = bushtel.merge(
    abs_iloc[
        [
            "match_key",
            "ILOC_CODE_2021",
            "iloc_name",
            "abs_total_population_2021",
            "abs_indigenous_population_2021"
        ]
    ],
    on="match_key",
    how="left",
    validate="many_to_one"
)


# ==================================================
# 8. IDENTIFY POTENTIAL ABS POPULATION MATCHES
# ==================================================

abs_candidate = (
    result["population_count"].isna()
    & result["abs_total_population_2021"].notna()
)

# Do not use the Acacia-Larrakia ILOC population for
# the ACACIA LARRAKIA Family Outstation.
# The ILOC may cover a much broader geographic area.
rejected_match = (
    abs_candidate
    & result["community_name"].str.upper().eq("ACACIA LARRAKIA")
)

accepted_abs_match = abs_candidate & ~rejected_match


# ==================================================
# 9. CREATE THE FINAL POPULATION COLUMN
# ==================================================

# Use the existing BushTel population first
result["population_final"] = result["population_count"]

# Use ABS population only when BushTel is missing
# and the exact-name match has been accepted
result.loc[
    accepted_abs_match,
    "population_final"
] = result.loc[
    accepted_abs_match,
    "abs_total_population_2021"
]


# ==================================================
# 10. RECORD THE POPULATION SOURCE AND MATCH STATUS
# ==================================================

result["population_final_source"] = "Unavailable"
result["population_match_status"] = "No population found"

# Records with an existing BushTel population
has_bushtel_population = result["population_count"].notna()

result.loc[
    has_bushtel_population,
    "population_final_source"
] = result.loc[
    has_bushtel_population,
    "population_source"
].fillna("BushTel")

result.loc[
    has_bushtel_population,
    "population_match_status"
] = "BushTel population"

# Records supplemented using ABS population
result.loc[
    accepted_abs_match,
    "population_final_source"
] = "ABS Census 2021 ILOC"

result.loc[
    accepted_abs_match,
    "population_match_status"
] = "ABS exact-name match - review recommended"

# Rejected matches
result.loc[
    rejected_match,
    "population_final_source"
] = "Unavailable"

result.loc[
    rejected_match,
    "population_match_status"
] = "Rejected - ILOC area may exceed outstation"


# ==================================================
# 11. REMOVE THE TEMPORARY MATCHING COLUMN
# ==================================================

result = result.drop(columns=["match_key"])


# ==================================================
# 12. EXPORT THE FINAL DATASET
# ==================================================

result.to_csv(
    "bushtel_with_abs_population_corrected.csv",
    index=False
)


# ==================================================
# 13. DISPLAY THE RESULTS
# ==================================================

print("Total BushTel records:", len(result))

print(
    "Records with BushTel population:",
    result["population_count"].notna().sum()
)

print(
    "Population records added from ABS:",
    accepted_abs_match.sum()
)

print(
    "Rejected ABS matches:",
    rejected_match.sum()
)

print(
    "Records with final population:",
    result["population_final"].notna().sum()
)

print(
    "Records still missing population:",
    result["population_final"].isna().sum()
)


# Display communities supplemented using ABS
abs_added = result[
    result["population_final_source"] == "ABS Census 2021 ILOC"
][
    [
        "community_name",
        "community_type",
        "iloc_name",
        "abs_total_population_2021",
        "population_final",
        "population_match_status"
    ]
]

print("\nCommunities supplemented using ABS population:")
print(abs_added.to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: '2021Census_geog_desc_1st_2nd_3rd_release(1).xlsx'